## Narrative Position

**Pipeline step:** Data lineage (1/6) — 표본 구성과 식별자 체인

**This notebook answers:** *"우리가 분석한 firm-year가 어떻게 수집되었고, 어디서 누락되었는가?"*

**Next:** `01_section_passage_v2.ipynb`

**Linked robustness evidence:** 없음 — lineage는 robustness 대상이 아니라 robustness의 *전제*다.

**Evaluation rubric coverage:** 데이터 이해 · DART lineage · 재현성

**Why this matters for our team's framing:** instability를 주장하려면 *어떤 표본 위에서 그 instability가 관찰되었는지*가 명확해야 한다. 이 notebook은 그 출발점을 고정한다.

> 전체 narrative 흐름은 `00_NARRATIVE_README.md`를 참조한다.

---


# 00 · Ingest v2 — 수집 lineage 검증

**Input**  :  (기존 수집 결과)
           + 
**Output** : 
           + 

---

## 이 단계가 답하는 한 줄

> "우리가 분석한 firm-year가 어떻게 수집되었고, 어디서 누락되었는가?"

**5-key lineage 원칙:**

lineage가 깨지면(company_name으로 join 등) 조용한 오매칭이 발생한다.

**주요 작업:**
1. stock_code zfill(6) 정규화
2. rcept_no 14자리 검증
3. 섹션 추출 성공/실패 분류
4. KCGS merge 준비 — NaN은 0으로 채우지 않음
5. salvage_log.csv 생성 (N=243 → N=210 손실 추적)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline_config import (
    D_META, D_RAW, D_SECTIONS, D_PASSAGES, D_MERGED,
    ID_COLS, RANDOM_SEED, GRADE_TO_7
)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"D_RAW: {D_RAW}")

## 1. Collected Reports 로드 + stock_code 정규화

In [ ]:
# collected_reports.csv: 01_collect.py 결과
collected_path = D_RAW / "collected_reports.csv"
assert collected_path.exists(), f"파일 없음: {collected_path}"

collected = pd.read_csv(
    collected_path,
    dtype={"stock_code": str, "corp_code": str, "rcept_no": str}
)

# stock_code zfill(6) — CSV 로드 시 앞자리 0 손실 방지
collected["stock_code"] = collected["stock_code"].astype(str).str.zfill(6)

print(f"Collected reports: {collected.shape}")
print(f"Columns: {collected.columns.tolist()}")
print(f"
unique stock_code: {collected["stock_code"].nunique()}")
print(f"fiscal_year dist: {collected["fiscal_year"].value_counts().sort_index().to_dict()}")
if "status" in collected.columns:
    print(f"
status dist:")
    print(collected["status"].value_counts())
display(collected.head(5))

## 2. rcept_no 검증 — 14자리 확인

In [ ]:
# rcept_no는 14자리 숫자 문자열이어야 함
# 예: 20220308000798
if "rcept_no" in collected.columns:
    rcept_valid = collected["rcept_no"].dropna()
    n_14digit = rcept_valid.apply(lambda x: len(str(x).replace(".0",""))==14).sum()
    n_nan     = collected["rcept_no"].isna().sum()
    
    print(f"rcept_no 현황:")
    print(f"  전체: {len(collected)}")
    print(f"  14자리 유효: {n_14digit}")
    print(f"  NaN (수집 실패): {n_nan}")
    print(f"  문자열 이상: {len(rcept_valid)-n_14digit}")
    
    # 이상 케이스 표시
    abnormal = collected[collected["rcept_no"].notna() &
                         ~collected["rcept_no"].apply(
                             lambda x: len(str(x).replace(".0",""))==14
                         )]
    if len(abnormal) > 0:
        print(f"
⚠️ 이상 rcept_no {len(abnormal)}건:")
        display(abnormal[[c for c in ["stock_code","fiscal_year","rcept_no"] if c in abnormal.columns]])

## 3. 섹션 추출 현황

In [ ]:
sections_path = D_SECTIONS / "extracted_sections.csv"

if sections_path.exists():
    sections = pd.read_csv(sections_path,
                           dtype={"stock_code": str, "corp_code": str, "rcept_no": str})
    sections["stock_code"] = sections["stock_code"].astype(str).str.zfill(6)
    
    print(f"Extracted sections: {sections.shape}")
    print(f"
unique stock_code: {sections["stock_code"].nunique()}")
    if "section" in sections.columns:
        print(f"section distribution:")
        print(sections["section"].value_counts())
    if "text_length" in sections.columns:
        print(f"
text_length stats:")
        print(sections.groupby("section")["text_length"].agg(["mean","min","max"]).astype(int))
else:
    print(f"[WARNING] extracted_sections.csv 없음: {sections_path}")
    print("02_extract_sections_fixed.py 먼저 실행 필요")

## 4. Lineage Audit — Collected → Sections → Passages

In [ ]:
# collected_reports와 extracted_sections 비교
if sections_path.exists():
    # firm-year 단위로 집계
    sections_fw = sections.groupby(["stock_code","fiscal_year"]).agg(
        n_sections=("section","count"),
        sections_present=("section",lambda x: "|".join(sorted(x.unique()))),
    ).reset_index()
    
    # collected와 left join
    lineage = collected[[c for c in ["stock_code","corp_code","rcept_no",
                                      "fiscal_year","esg_year","status"]
                          if c in collected.columns]].copy()
    
    lineage = lineage.merge(sections_fw, on=["stock_code","fiscal_year"], how="left")
    lineage["section_extracted"] = lineage["n_sections"].notna()
    
    print(f"Lineage audit ({len(lineage)} firm-year):")
    print(f"  sections 추출 성공: {lineage["section_extracted"].sum()}")
    print(f"  sections 추출 실패: {(~lineage["section_extracted"]).sum()}")
    
    # KCGS merge 확인
    merged_path = D_MERGED / "merged_exp_F.csv"
    if merged_path.exists():
        merged = pd.read_csv(merged_path, dtype={"stock_code": str})
        merged["stock_code"] = merged["stock_code"].astype(str).str.zfill(6)
        n_merged = len(merged)
        n_kcgs_valid = merged["kcgs_grade"].notna().sum() if "kcgs_grade" in merged.columns else 0
        print(f"
KCGS merge 현황:")
        print(f"  merged rows: {n_merged}")
        print(f"  KCGS 유효: {n_kcgs_valid}")
        print(f"  KCGS NaN (미평가/신규상장): {n_merged-n_kcgs_valid}")
    
    # 저장
    audit_path = D_META / "lineage_audit_v2.csv"
    lineage.to_csv(audit_path, index=False, encoding="utf-8-sig")
    print(f"
→ saved: {audit_path}")

## 5. Salvage Log — 수집 실패 유형 분류

In [ ]:
import datetime

# salvage_log: 각 firm-year의 최종 상태 기록
salvage_rows = []

for _, row in collected.iterrows():
    # rcept_no 상태
    rcept_str = str(row.get("rcept_no","")).replace(".0","")
    has_rcept = len(rcept_str) == 14
    
    # KCGS 상태 (merged에서 조회)
    kcgs_ok = None
    if "merged" in dir() and len(merged) > 0:
        merged_row = merged[(merged["stock_code"]==row["stock_code"]) &
                            (merged["fiscal_year"]==row["fiscal_year"])]
        if len(merged_row) > 0 and "kcgs_grade" in merged_row.columns:
            kcgs_ok = merged_row.iloc[0]["kcgs_grade"] not in [None, "미공개",""] \n                      and pd.notna(merged_row.iloc[0]["kcgs_grade"])
    
    status_code = row.get("status","unknown")
    if not has_rcept:
        final_decision = "EXCLUDE_NO_RCEPT"
    elif kcgs_ok is False:
        final_decision = "EXCLUDE_KCGS_NA"
    elif kcgs_ok:
        final_decision = "INCLUDE"
    else:
        final_decision = "UNKNOWN"
    
    salvage_rows.append({
        "stock_code":       row.get("stock_code",""),
        "fiscal_year":      row.get("fiscal_year",""),
        "corp_code":        row.get("corp_code",""),
        "rcept_no":         rcept_str if has_rcept else "MISSING",
        "has_rcept":        has_rcept,
        "status":           status_code,
        "kcgs_valid":       kcgs_ok,
        "final_decision":   final_decision,
        "audit_ts":         datetime.datetime.now().isoformat(),
    })

salvage_df = pd.DataFrame(salvage_rows)
print("=== Salvage Log Summary ===")
print(salvage_df["final_decision"].value_counts())

salvage_path = D_META / "salvage_log.csv"
salvage_df.to_csv(salvage_path, index=False, encoding="utf-8-sig")
print(f"
→ saved: {salvage_path}")

## 6. 표본 정의 요약 (보고서 Table 1)

이 표가 3장 데이터 절에 그대로 들어간다.

In [ ]:
summary = {
    "단계": [
        "KCGS 평가 대상 KOSPI 기업 (company_master)",
        "OpenDART 사업보고서 수집 성공",
        "섹션 추출 (II/IV/VI) 성공",
        "ESG passages 필터링 후",
        "KCGS merge 후 회귀 표본"
    ],
    "N_firm_year": [
        len(collected) if "collected" in dir() else "?",
        collected["status"].eq("success").sum() if ("collected" in dir() and "status" in collected.columns) else "?",
        sections["stock_code"].nunique() * 3 if sections_path.exists() else "?",
        "N/A",
        n_kcgs_valid if "n_kcgs_valid" in dir() else "?"
    ],
    "비고": [
        "3개 연도 (2021/2022/2023)",
        "팀원C N=243 (N=81×3)",
        "섹션 없는 케이스 별도 로그",
        "esg_passages.csv",
        "KCGS 미평가 제외, NaN 보존"
    ]
}

summary_df = pd.DataFrame(summary)
display(summary_df)

# HTML 테이블 저장 (보고서용)
html_path = str(D_META / "sample_definition_table.html")
summary_df.to_html(html_path, index=False, border=1)
print(f"→ saved: {html_path}")

## 7. 다음 단계

**확인 후 진행:**
- lineage_audit_v2.csv에서 섹션 추출 실패 케이스 사유 확인
- salvage_log.csv에서 EXCLUDE 케이스 검토 (0으로 채우지 않음 원칙 준수 확인)

**다음 단계:** 
- II/IV/VI 섹션 텍스트에서 ESG passage 필터링
- sentence_density_filter 적용
- passage_filter_log.csv 생성